# Quantum Machine Learning (QML) — Python Laboratory
### A compact, executable introduction to quantum-enhanced machine learning

**Goal:** understand what quantum computing may contribute to machine learning, then run small, commented Python demonstrations.

> **Reality check:** quantum computing is not a drop-in faster replacement for GPUs. Advantages are problem- and algorithm-specific. This notebook uses local simulators and small datasets; simulator timings are not evidence of quantum-hardware speedup.

**Learning path**
1. Why QML? — potential advantages and where they may fit in an ML lifecycle.
2. Quantum fundamentals — qubits, gates, entanglement, measurement.
3. Algorithm gallery — concise explanations and runnable demonstrations.
4. Classical comparison — evaluate quality and end-to-end runtime fairly.
5. Research notes — distinguish theoretical, experimental, and unproven claims.

**Primary framework:** PennyLane. Classical comparisons use NumPy and scikit-learn. Optional PyTorch integration can be added later without making it a notebook requirement.

## 1. Why consider quantum machine learning?

Potential reasons to investigate QML:

- **Specialized computational speedups:** some quantum algorithms have theoretical advantages for particular structured problems.
- **New representations:** quantum states and Hilbert-space feature maps provide alternative ways to encode and compare data.
- **Optimization:** variational circuits and QAOA offer different approaches to selected optimization problems.
- **Sampling:** quantum measurement naturally produces samples from quantum probability distributions.
- **Quantum-native simulation:** molecules, materials, and other quantum systems can be represented more naturally.
- **Pattern discovery:** quantum kernels and variational circuits may help on selected classification or anomaly-detection tasks.
- **Generative modeling:** quantum circuits can parameterize probability distributions for generative experiments.
- **Hybrid architectures:** classical CPUs/GPUs can handle data and large tensor operations while quantum processors handle suitable subroutines.
- **New learning methods:** quantum circuits introduce different model families, inductive biases, and optimization behavior.

### Where QML may fit in an ML lifecycle

| Lifecycle stage | Possible quantum contribution | Example |
|---|---|---|
| Data representation | Quantum feature encoding | Feature maps |
| Feature analysis | Quantum similarity/kernel calculation | Quantum kernel |
| Model development | Quantum/hybrid model architecture | QNN, VQC |
| Optimization | Specialized combinatorial optimization | QAOA |
| Sampling/generation | Quantum probability distributions | QBM, QGAN |
| Simulation | Quantum-system simulation | VQE, quantum chemistry |
| Evaluation | Compare learned representations and distributions | Kernel/classifier benchmarks |

**Important limits:** encoding classical data, circuit noise, measurement shots, limited qubits, error correction, and data-transfer overhead can erase a theoretical advantage. A quantum state with \(2^n\) amplitudes does not mean we can read out \(2^n\) values in one measurement.

## 2. Quantum fundamentals

A **qubit** can be in a superposition of basis states. For one qubit:

\[
|\psi\rangle = \alpha|0\rangle + \beta|1\rangle, \quad
|\alpha|^2 + |\beta|^2 = 1
\]

A **gate** changes a quantum state. An **entangling gate** can create correlations that cannot be described as independent states. **Measurement** returns classical outcomes sampled according to probabilities.

The next cell installs the small set of libraries used here. In hosted notebooks, run it once if packages are missing.

In [ ]:
%pip -q install pennylane scikit-learn matplotlib pandas

In [ ]:
import sys, time, platform, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import pennylane as qml
    from sklearn.datasets import make_moons, make_classification
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import StandardScaler
    from sklearn.svm import SVC
    from sklearn.metrics import accuracy_score
except ImportError as exc:
    raise ImportError("Run the installation cell above, then rerun this cell.") from exc

SEED = 42
rng = np.random.default_rng(SEED)
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("PennyLane:", qml.__version__)
print("NumPy:", np.__version__)

### A first circuit: superposition and measurement

The Hadamard gate creates an equal superposition from \(|0\rangle\). Repeated measurements should be approximately 50/50.

In [ ]:
dev = qml.device("default.qubit", wires=1, shots=1000)

@qml.qnode(dev)
def hadamard_demo():
    qml.Hadamard(wires=0)
    return qml.sample(wires=0)

samples = hadamard_demo()
values, counts = np.unique(samples, return_counts=True)
print(dict(zip(values.astype(int), counts)))
plt.bar(values.astype(str), counts / len(samples))
plt.xlabel("Measured bit")
plt.ylabel("Observed frequency")
plt.title("Hadamard superposition: finite-shot measurement")
plt.ylim(0, 1)
plt.show()

### Entanglement

A Hadamard followed by a CNOT creates a Bell state. Measuring both qubits yields correlated outcomes (00 or 11).

In [ ]:
dev_bell = qml.device("default.qubit", wires=2, shots=1000)

@qml.qnode(dev_bell)
def bell_pair():
    qml.Hadamard(wires=0)
    qml.CNOT(wires=[0, 1])
    return qml.sample(wires=[0, 1])

bell_samples = bell_pair()
unique, counts = np.unique(bell_samples, axis=0, return_counts=True)
print({"".join(map(str, row)): int(n) for row, n in zip(unique, counts)})

## 3. Algorithm gallery

The gallery below separates **actual runnable circuit demonstrations** from algorithm families that need additional assumptions or specialized implementations. The goal is to show the computational idea without implying every named method has a practical speed advantage.

Each entry gives its purpose, a brief explanation, and (where practical) a compact executable demonstration.

### 3.1 Quantum feature map and quantum kernel

**Purpose:** encode classical feature vectors into quantum states, then compare two inputs using state overlap. A quantum kernel can be passed to a classical SVM.

**Why it matters:** quantum feature maps define alternative similarity functions. Whether they outperform classical kernels depends on the dataset, encoding, hardware, and cost of evaluating the kernel.

In [ ]:
n_qubits = 2
dev_kernel = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev_kernel)
def feature_state(x):
    # Encode two standardized features as rotation angles.
    for wire in range(n_qubits):
        qml.RY(float(x[wire]), wires=wire)
    # Entangle features so the representation is not a product of independent rotations.
    qml.CNOT(wires=[0, 1])
    qml.RZ(float(x[0] * x[1]), wires=1)
    return qml.state()

def quantum_kernel(x1, x2):
    # Fidelity between pure states: |<phi(x1)|phi(x2)>|^2.
    a, b = feature_state(x1), feature_state(x2)
    return float(np.abs(np.vdot(a, b)) ** 2)

X, y = make_moons(n_samples=120, noise=0.18, random_state=SEED)
X = StandardScaler().fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=SEED, stratify=y
)
K_train = np.array([[quantum_kernel(a, b) for b in X_train] for a in X_train])
K_test = np.array([[quantum_kernel(a, b) for b in X_train] for a in X_test])

qsvm = SVC(kernel="precomputed").fit(K_train, y_train)
qsvm_pred = qsvm.predict(K_test)
print("Quantum-kernel SVM accuracy:", round(accuracy_score(y_test, qsvm_pred), 3))

### 3.2 Variational Quantum Classifier (VQC) / Quantum Neural Network (QNN)

**Purpose:** train adjustable quantum-circuit parameters to classify data.

**How it works:** encode features, apply trainable rotations and entangling gates, measure an expectation value, then update circuit parameters with a classical optimizer. This is a **hybrid** quantum-classical training loop.

In [ ]:
n_wires = 2
dev_vqc = qml.device("default.qubit", wires=n_wires)

def circuit_block(weights):
    # Trainable local rotations followed by entanglement.
    for w in range(n_wires):
        qml.RY(weights[w, 0], wires=w)
        qml.RZ(weights[w, 1], wires=w)
    qml.CNOT(wires=[0, 1])

@qml.qnode(dev_vqc, interface="autograd")
def vqc(x, weights):
    # Data encoding.
    for w in range(n_wires):
        qml.RY(x[w], wires=w)
    circuit_block(weights)
    return qml.expval(qml.PauliZ(0))

# Use a small, balanced subset to keep simulator training quick.
X_small, y_small = X_train[:48], y_train[:48]
y_pm = 2 * y_small - 1
weights = qml.numpy.array(rng.normal(0, 0.1, size=(n_wires, 2)), requires_grad=True)
opt = qml.GradientDescentOptimizer(stepsize=0.15)

def loss_fn(w):
    predictions = qml.numpy.stack([vqc(x, w) for x in X_small])
    return qml.numpy.mean((predictions - y_pm) ** 2)

start = time.perf_counter()
loss_history = []
for epoch in range(25):
    weights = opt.step(loss_fn, weights)
    loss_history.append(float(loss_fn(weights)))
elapsed_vqc = time.perf_counter() - start

vqc_scores = np.array([float(vqc(x, weights)) for x in X_test])
vqc_pred = (vqc_scores < 0).astype(int)  # sign maps expectation to class
print("VQC accuracy:", round(accuracy_score(y_test, vqc_pred), 3))
print("Simulator training seconds:", round(elapsed_vqc, 3))
plt.plot(loss_history)
plt.xlabel("Epoch")
plt.ylabel("Mean squared error")
plt.title("VQC training loss (small simulator experiment)")
plt.show()

### 3.3 Quantum Approximate Optimization Algorithm (QAOA)

**Purpose:** approximate solutions to selected combinatorial optimization problems.

**How it works:** alternate a problem-dependent cost operation and a mixing operation, controlled by variational angles. A classical optimizer searches for angles that improve the measured objective.

This small example demonstrates QAOA-style optimization for a two-node MaxCut graph. It is a toy simulator demonstration, not evidence that QAOA beats classical optimization.

In [ ]:
# Two-node graph with one edge. MaxCut is maximized when the bits differ.
dev_qaoa = qml.device("default.qubit", wires=2, shots=2000)

@qml.qnode(dev_qaoa)
def qaoa_maxcut(gamma, beta):
    qml.Hadamard(0); qml.Hadamard(1)
    # Cost unitary for the single edge (0, 1).
    qml.IsingZZ(2 * gamma, wires=[0, 1])
    # Mixer unitary.
    qml.RX(2 * beta, wires=0); qml.RX(2 * beta, wires=1)
    return qml.probs(wires=[0, 1])

def expected_cut(params):
    probs = qaoa_maxcut(params[0], params[1])
    # Basis order 00, 01, 10, 11; middle states cut the edge.
    return probs[1] + probs[2]

grid = np.linspace(0, np.pi, 31)
best = max(((float(expected_cut([g, b])), g, b) for g in grid for b in grid))
print({"estimated_cut_probability": round(best[0], 3), "gamma": round(best[1], 3), "beta": round(best[2], 3)})

### 3.4 Grover search / amplitude amplification

**Purpose:** amplify the probability of a marked item in an unstructured search space.

**Why it matters:** Grover's algorithm gives a quadratic query improvement in the ideal oracle model (roughly \(O(\sqrt{N})\) queries versus \(O(N)\)). This is not a general speedup for arbitrary ML training.

In [ ]:
dev_grover = qml.device("default.qubit", wires=2)

@qml.qnode(dev_grover)
def grover_two_qubit():
    # Uniform superposition over four candidates.
    qml.Hadamard(0); qml.Hadamard(1)
    # Oracle marks |11> by flipping its phase.
    qml.PauliX(0); qml.PauliX(1)
    qml.Hadamard(1); qml.Toffoli(wires=[0, 1, 1]) if False else None
    # For this tiny example, use a controlled phase on |11>.
    qml.CZ(wires=[0, 1])
    qml.PauliX(0); qml.PauliX(1)
    # Diffusion operator.
    qml.Hadamard(0); qml.Hadamard(1)
    qml.PauliZ(0); qml.PauliZ(1); qml.CZ(wires=[0, 1])
    qml.Hadamard(0); qml.Hadamard(1)
    return qml.probs(wires=[0, 1])

print("Output probabilities (basis order 00, 01, 10, 11):", np.round(grover_two_qubit(), 3))

### 3.5 The remaining algorithm families

These are included in the notebook's map so the terminology and intended use are clear. Some are research prototypes or broad families rather than one universally standardized algorithm. Their practical implementation depends on assumptions such as data access, circuit model, and hardware.

### 3.6 Quantum K-Means

**What it does:** Quantum-assisted clustering / distance estimation. Potential use: clustering when the data representation and distance subroutine suit a quantum formulation.

**Practical note:** Research-dependent; classical k-means is the baseline.

### 3.7 Quantum Principal Component Analysis (qPCA)

**What it does:** Quantum algorithms for estimating dominant components of density matrices under state-preparation and sampling assumptions.

**Practical note:** Not a drop-in replacement for sklearn PCA on ordinary CSV data.

### 3.8 Quantum Linear Regression / Quantum Linear Solver (QLR / QLS)

**What it does:** Quantum linear-system approaches (including HHL-style methods) can offer theoretical advantages under restrictive matrix and output assumptions.

**Practical note:** Input loading, condition number, sparsity, and readout costs matter.

### 3.9 Quantum Boltzmann Machine (QBM)

**What it does:** Energy-based probabilistic model using quantum states or quantum sampling to represent distributions.

**Practical note:** Training and sampling remain challenging.

### 3.10 Quantum Generative Adversarial Network (QGAN)

**What it does:** Hybrid generator/discriminator setup where a parameterized quantum circuit can generate samples.

**Practical note:** Compare sample quality and total training cost with classical GANs.

### 3.11 Quantum Reinforcement Learning (QRL)

**What it does:** Use quantum circuits or quantum-enhanced subroutines in an agent's policy, value function, or environment model.

**Practical note:** A research family; advantage depends on the task and access model.

### 3.12 Quantum Discriminant Analysis

**What it does:** Quantum-inspired or quantum algorithms for discriminant projections and class separation.

**Practical note:** Terminology covers multiple formulations; validate the chosen paper/implementation.

### 3.13 Quantum Naive Bayes

**What it does:** Quantum approaches to probabilistic classification and state-based representations.

**Practical note:** No general practical advantage established.

### 3.14 Quantum Ensemble Methods

**What it does:** Combine quantum classifiers or hybrid quantum/classical learners.

**Practical note:** Ensembling is a strategy, not one single canonical quantum algorithm.

### 3.15 Quantum Transfer Learning

**What it does:** Use a pretrained classical feature extractor with a small trainable quantum circuit head.

**Practical note:** A practical hybrid demonstration, but not inherently faster.

### 3.16 Quantum Feature Maps

**What it does:** Encode classical features into quantum states to define a kernel or learned representation.

**Practical note:** Demonstrated above as the foundation of a quantum-kernel SVM.

### 3.17 Quantum Autoencoders

**What it does:** Train quantum circuits to compress and reconstruct quantum states.

**Practical note:** Most natural when the data itself is quantum-state data.

### 3.18 Quantum Time-Series Models

**What it does:** Apply quantum circuits to sequence features, forecasting, or temporal classification.

**Practical note:** Active research; compare against classical time-series baselines.

### 3.19 Variational Quantum Eigensolver (VQE)

**What it does:** Hybrid quantum-classical method for estimating low-energy eigenvalues of quantum systems.

**Practical note:** Especially relevant to quantum chemistry and materials, rather than ordinary tabular ML.

### 3.20 Quantum Approximate Optimization (QAOA)

**What it does:** Variational circuit for approximate combinatorial optimization.

**Practical note:** Runnable two-node MaxCut example above.

## 4. Classical baseline and fair benchmarking

A useful benchmark compares equivalent tasks, data splits, quality targets, and end-to-end costs. The quantum-kernel example above can be compared with an RBF SVM using the same train/test split.

In [ ]:
# Classical RBF SVM baseline on the exact same data split.
start = time.perf_counter()
classical_svm = SVC(kernel="rbf", gamma="scale").fit(X_train, y_train)
classical_fit_seconds = time.perf_counter() - start
classical_pred = classical_svm.predict(X_test)

results = pd.DataFrame([
    {"model": "Classical RBF SVM", "test_accuracy": accuracy_score(y_test, classical_pred),
     "fit_seconds": classical_fit_seconds, "measurement": "CPU fit only"},
    {"model": "Quantum-kernel SVM", "test_accuracy": accuracy_score(y_test, qsvm_pred),
     "fit_seconds": np.nan, "measurement": "Kernel construction excluded from fit timing"},
    {"model": "Small VQC", "test_accuracy": accuracy_score(y_test, vqc_pred),
     "fit_seconds": elapsed_vqc, "measurement": "Simulator training loop"}
])
display(results)

### Benchmark interpretation

- The quantum-kernel SVM's kernel construction time is deliberately shown as missing above because it was computed in an earlier cell. Do not compare its SVM fit time alone with a full classical fit.
- The VQC runs on a classical simulator. Its wall-clock time measures simulation and Python overhead, not quantum hardware execution.
- For a hardware experiment, include queue time, circuit compilation, shots, network/API overhead, measurement, and classical post-processing.
- Accuracy on a small synthetic dataset is a functional check, not evidence of generalization or quantum advantage.

In [ ]:
# Measure quantum kernel construction explicitly for a smaller subset.
Xb = X_train[:30]
start = time.perf_counter()
Kb = np.array([[quantum_kernel(a, b) for b in Xb] for a in Xb])
kernel_build_seconds = time.perf_counter() - start
print(f"Quantum kernel matrix (30 x 30) construction: {kernel_build_seconds:.3f} seconds")

## 5. Takeaways

- Quantum computing offers a different computational model, not a universal accelerator.
- The clearest motivations include specialized optimization, quantum-native simulation, sampling, and alternative feature representations.
- Hybrid quantum-classical learning is the most accessible development pattern today.
- Claims of speedup must include data loading, circuit execution, measurement, communication, and output extraction.
- A quantum advantage is meaningful only when the quantum method beats the best relevant classical method on the same task and quality target.

### Further reading
- PennyLane documentation: https://docs.pennylane.ai/
- IBM Quantum learning: https://learning.quantum.ibm.com/
- Qiskit documentation: https://docs.quantum.ibm.com/
- Review the assumptions and resource estimates in any claimed quantum-advantage paper before interpreting its headline result.